<a href="https://colab.research.google.com/github/Arsen645/Leaders-background-analysis/blob/main/Leaders_Education_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install country_converter -q

import pandas as pd
import country_converter as coco

In [20]:
mad = pd.read_excel("mpd2023_web.xlsx", sheet_name="Full data")
mad = mad[["countrycode", "country", "year", "gdppc"]]
mad = mad.sort_values(["countrycode", "year"])

mad["gdppc_lag"] = mad.groupby("countrycode")["gdppc"].shift(1)
mad["prev_year"] = mad.groupby("countrycode")["year"].shift(1)
mad.loc[mad["year"] - mad["prev_year"] != 1, "gdppc_lag"] = None
mad["growth"] = (mad["gdppc"] / mad["gdppc_lag"] - 1) * 100
mad.head()

,countrycode,country,year,gdppc,gdppc_lag,prev_year,growth
0,AFG,Afghanistan,1,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,730,NaN,NaN,1.0,NaN
2,AFG,Afghanistan,1000,NaN,NaN,730.0,NaN
3,AFG,Afghanistan,1090,NaN,NaN,1000.0,NaN
4,AFG,Afghanistan,1150,NaN,NaN,1090.0,NaN


In [13]:
codes = mad["countrycode"].unique()
cow = coco.convert(list(codes), src="ISO3", to="GWcode", not_found=None)
mapping = dict(zip(codes, cow))

# add the old countries by hand
mapping.update({"SUN": 365, "YUG": 345, "CSK": 315})

mad["ccode"] = pd.to_numeric(mad["countrycode"].map(mapping), errors="coerce")

# Gleditsch-Ward → Correlates of War fixes
mad.loc[mad["ccode"] == 260, "ccode"] = 255    # Germany
mad.loc[mad["ccode"] == 340, "ccode"] = 345    # Serbia
mad.loc[(mad["countrycode"] == "YEM") & (mad["year"] >= 1990), "ccode"] = 679  # unified Yemen

# old country until break-up, new one after
mad = mad[~((mad["countrycode"] == "SUN") & (mad["year"] > 1991))]
mad = mad[~((mad["countrycode"] == "RUS") & (mad["year"] <= 1991))]
mad = mad[~((mad["countrycode"] == "YUG") & (mad["year"] > 1991))]
mad = mad[~((mad["countrycode"] == "SRB") & (mad["year"] <= 1991))]

print("Not matched:", mad.loc[mad["ccode"].isna(), "country"].unique())
mad = mad.dropna(subset=["ccode"])
mad["ccode"] = mad["ccode"].astype(int)
print("Duplicates:", mad.duplicated(["ccode", "year"]).sum())

Not matched: ['China, Hong Kong SAR' 'Puerto Rico' 'State of Palestine']
Duplicates: 776


In [16]:
lead = pd.read_stata("leaders_datapaper_replication_final_9_10_15.dta",
                     convert_categoricals=False)
print(lead.shape)

(2965, 83)


In [17]:
keep = ["ccode", "year", "leaderid", "leadername", "leveledu"]
lead = lead[keep]
lead.head()

,ccode,year,leaderid,leadername,leveledu
0,2,1877,LEAD.v1-1,Grant,2.0
1,2,1881,LEAD.v1-4,Hayes,3.0
2,2,1881,LEAD.v1-7,Garfield,2.0
3,2,1885,LEAD.v1-10,C. Arthur,3.0
4,2,1889,LEAD.v1-13,Cleveland,2.0


In [18]:
lead = lead.drop_duplicates(subset=["ccode", "year"], keep="last")
lead["ccode"] = lead["ccode"].astype(int)
print(lead.shape)

(2393, 5)


In [21]:
cc = coco.CountryConverter()
print([c for c in cc.data.columns if "cow" in c.lower() or "gw" in c.lower()])

['GWcode']


In [22]:
codes = mad["countrycode"].unique()
cow = coco.convert(list(codes), src="ISO3", to="GWcode", not_found=None)
mapping = dict(zip(codes, cow))
mad["ccode"] = pd.to_numeric(mad["countrycode"].map(mapping), errors="coerce")

print("Not matched:", mad.loc[mad["ccode"].isna(), "country"].unique())
mad = mad.dropna(subset=["ccode"])
mad["ccode"] = mad["ccode"].astype(int)

Not matched: ['Czechoslovakia' 'China, Hong Kong SAR' 'Puerto Rico'
 'State of Palestine' 'Former USSR' 'Former Yugoslavia']


In [23]:
df = lead.merge(mad[["ccode", "country", "year", "gdppc", "gdppc_lag", "growth"]],
                on=["ccode", "year"], how="inner")
print(df.shape)
df.head()

(2240, 9)


,ccode,year,leaderid,leadername,leveledu,country,gdppc,gdppc_lag,growth
0,2,1877,LEAD.v1-1,Grant,2.0,United States,5098.6896,5049.8368,0.967413
1,2,1881,LEAD.v1-7,Garfield,2.0,United States,6317.4384,6255.7296,0.986436
2,2,1885,LEAD.v1-10,C. Arthur,3.0,United States,6424.1432,6523.1344,-1.517540
3,2,1889,LEAD.v1-13,Cleveland,2.0,United States,6705.6896,6447.2840,4.007976
4,2,1893,LEAD.v1-16,Harrison,2.0,United States,6834.2496,7324.0632,-6.687730


In [24]:
print("Before:", len(df))

df = df.dropna(subset=["growth", "leveledu"])   # delete row if no growth or no education
df = df[df["leveledu"] >= 0]                     # remove negative "missing" codes, if any
df = df[(df["year"] >= 1875) & (df["year"] <= 2004)]

print("After:", len(df))

Before: 2240
After: 1894


In [25]:
from google.colab import files
files.download("leaders_gdp.csv")

FileNotFoundError: Cannot find file: leaders_gdp.csv

In [26]:
df = df[["country", "ccode", "year", "leaderid", "leadername", "leveledu", "gdppc", "gdppc_lag", "growth"]]
print(df.describe())

df.to_csv("leaders_gdp.csv", index=False)
files.download("leaders_gdp.csv")

             ccode         year     leveledu         gdppc     gdppc_lag  \
count  1894.000000  1894.000000  1894.000000   1894.000000   1894.000000   
mean    342.993664  1956.010560     2.279831   7839.404098   7703.759552   
std     238.637841    35.603882     0.788751   8429.770154   8209.074452   
min       2.000000  1875.000000     0.000000    377.580086    377.580086   
25%     155.000000  1929.000000     2.000000   2381.500000   2381.500000   
50%     235.000000  1963.000000     2.000000   4810.500000   4771.932084   
75%     475.000000  1988.000000     3.000000   9313.961565   9196.250000   
max     920.000000  2001.000000     3.000000  62416.000000  59731.000000   

            growth  
count  1894.000000  
mean      1.294977  
std       7.364055  
min     -58.446661  
25%      -1.050850  
50%       1.846468  
75%       4.409485  
max      79.445322  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
df.head()

,country,ccode,year,leaderid,leadername,leveledu,gdppc,gdppc_lag,growth
0,United States,2,1877,LEAD.v1-1,Grant,2.0,5098.6896,5049.8368,0.967413
1,United States,2,1881,LEAD.v1-7,Garfield,2.0,6317.4384,6255.7296,0.986436
2,United States,2,1885,LEAD.v1-10,C. Arthur,3.0,6424.1432,6523.1344,-1.517540
3,United States,2,1889,LEAD.v1-13,Cleveland,2.0,6705.6896,6447.2840,4.007976
4,United States,2,1893,LEAD.v1-16,Harrison,2.0,6834.2496,7324.0632,-6.687730


In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv("leaders_gdp.csv")

# keep only rows with everything we need
df = df.dropna(subset=["growth", "leveledu", "gdppc_lag", "ccode", "year"])

# log of starting income (standard in growth research)
df["log_gdppc_lag"] = np.log(df["gdppc_lag"])

# remove extreme growth values (wars, data errors)
df = df[(df["growth"] > -30) & (df["growth"] < 30)]

print(df.shape)
df["leveledu"].value_counts()

(1879, 10)


,count
leveledu,
3.0,861
2.0,748
1.0,208
0.0,62


In [30]:
import statsmodels.formula.api as smf

model = smf.ols(
    "growth ~ C(leveledu) + log_gdppc_lag + C(ccode) + C(year)",
    data=df
).fit(cov_type="cluster",
      cov_kwds={"groups": pd.factorize(df["ccode"])[0]})

# show only the interesting rows
print(model.summary().tables[1].as_text()[:1500])
print("R-squared:", round(model.rsquared, 3))

                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              7.8895      7.954      0.992      0.321      -7.701      23.479
C(leveledu)[T.1.0]    -0.6443      0.852     -0.757      0.449      -2.313       1.025
C(leveledu)[T.2.0]    -0.5697      0.787     -0.723      0.469      -2.113       0.974
C(leveledu)[T.3.0]    -0.8710      0.788     -1.105      0.269      -2.416       0.674
C(ccode)[T.20]        -1.6518      0.550     -3.004      0.003      -2.730      -0.574
C(ccode)[T.40]         1.8240      1.508      1.209      0.227      -1.133       4.780
C(ccode)[T.41]        -4.9945      2.436     -2.050      0.040      -9.769      -0.220
C(ccode)[T.42]        -2.0992      1.875     -1.119      0.263      -5.775       1.576
C(ccode)[T.51]        -1.7587      1.605     -1.096      0.273      -4.904       1.386
C(ccode)[T.52]        -0.9119      0.992   

/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 287, but rank is 102
  res = self.wald_test(


In [32]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = pd.get_dummies(df[["leveledu", "log_gdppc_lag", "year", "ccode"]],
                   columns=["ccode"], dtype=int)
y = df["growth"]

# time split: learn from the past, predict the future
train = df["year"] <= 1980
test = df["year"] > 1980
print("Train rows:", train.sum(), " Test rows:", test.sum())

Train rows: 1284  Test rows: 595


In [33]:
models = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                                           random_state=42, n_jobs=-1)
}
all_cols = list(X.columns)
no_edu = [c for c in all_cols if c != "leveledu"]

results = []
for name, m in models.items():
    for label, cols in [("with education", all_cols), ("without education", no_edu)]:
        m.fit(X.loc[train, cols], y[train])
        pred = m.predict(X.loc[test, cols])
        results.append({
            "Model": name, "Features": label,
            "RMSE": mean_squared_error(y[test], pred) ** 0.5,
            "MAE": mean_absolute_error(y[test], pred),
            "R2": r2_score(y[test], pred)
        })

pd.DataFrame(results).round(3)

,Model,Features,RMSE,MAE,R2
0,Linear regression,with education,6.134,4.266,-0.329
1,Linear regression,without education,6.129,4.256,-0.327
2,Random forest,with education,5.718,4.293,-0.155
3,Random forest,without education,5.715,4.287,-0.154


In [35]:
rf = models["Random forest"]
rf.fit(X.loc[train, all_cols], y[train])

imp = pd.Series(rf.feature_importances_, index=all_cols)
imp = imp[~imp.index.str.startswith("ccode_")]   # hide the country columns
print(imp.sort_values(ascending=False))

year             0.747008
log_gdppc_lag    0.231396
leveledu         0.020040
dtype: float64
